# Synthetic Burst Audit & MGEN-Feasibility Projection

This step is the **preprocessing step** between the statistical traffic generator and the MGEN scenario constructor.

```
burst generation from traces
        ↓
  ★ THIS NOTEBOOK ★   ← audit + project into MGEN-feasible bursts
        ↓
multi-UE scenario construction 
        ↓
.mgn export                     
        ↓
deployment / test              
```

## What it does

the synthetic bursts are **statistically valid** in the Markov model but not **directly executable** in MGEN.
Now I handle the difference:

1. Audits every known failure mode, distinguishing *statistical edge-cases* from *data corruption*
2. Projects edge-cases into MGEN-feasible equivalents, documenting every
   approximation explicitly
3. Applies a byte-preserving packet-count adjustment instead of clipping  packet size 
4. Estimates realistic UE capacity from the cleaned pool


## Cell 1 — Paths & Load

In [1]:
import math
import yaml
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional, Dict

# ── helpers ───────────────────────────────────────────────────────────────────

def find_project_root(start: Optional[Path] = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "data").is_dir() and (cand / "artifacts").is_dir():
            return cand
    raise FileNotFoundError(
        "Could not find project root containing both ./data and ./artifacts"
    )

def find_latest_run_dir(base_dir: Path) -> Path:
    if not base_dir.exists():
        raise FileNotFoundError(f"Base directory does not exist: {base_dir}")
    run_dirs = sorted(
        p for p in base_dir.iterdir()
        if p.is_dir() and p.name.startswith("run_")
    )
    if not run_dirs:
        raise FileNotFoundError(f"No run_* directories found in: {base_dir}")
    return run_dirs[-1]


# ── load app list from scenario_config.yaml ───────────────────────────────────
ROOT     = find_project_root()
ART      = ROOT / "artifacts"

_cfg_path = ROOT / "scenario_config.yaml"
if _cfg_path.exists():
    with open(_cfg_path) as _f:
        _cfg = yaml.safe_load(_f)
    APPS = [str(a).strip().lower().replace(" ", "_").replace("-", "_")
            for a in _cfg.get("apps", [])]
    print(f"  Apps loaded from scenario_config.yaml: {APPS}")
else:
    # fallback: discover all apps available in artifacts/
    APPS = sorted(
        d.name for d in ART.iterdir()
        if d.is_dir()
        and (d / "downlink").exists()
        and (d / "uplink").exists()
    )
    print(f"  scenario_config.yaml not found — discovered apps: {APPS}")

if not APPS:
    raise FileNotFoundError(
        "No apps found. Run the traffic generation pipeline first to "
        "populate artifacts/<app>/downlink/ and artifacts/<app>/uplink/."
    )

# ── load all apps ─────────────────────────────────────────────────────────────
# app_raw[app] = {"dl": DataFrame, "ul": DataFrame}
app_raw: Dict[str, Dict] = {}

print()
print("="*76)
print("  LOADING BURST POOLS")
print("="*76)

for app in APPS:
    app_dir = ART / app
    dl_run  = find_latest_run_dir(app_dir / "downlink")
    ul_run  = find_latest_run_dir(app_dir / "uplink")

    dl_path = dl_run / "data" / "synth_test_bursts_markov.parquet"
    ul_path = ul_run / "data" / "synth_test_bursts_markov.parquet"

    for p in [dl_path, ul_path]:
        if not p.exists():
            raise FileNotFoundError(f"Missing: {p}")

    dl = pd.read_parquet(dl_path)
    ul = pd.read_parquet(ul_path)
    app_raw[app] = {"dl": dl, "ul": ul}

    print(f"  📱 {app.upper()}")
    print(f"     DL: {len(dl):,} bursts  {dl['synthetic_flow_id'].nunique()} flows  "
          f"{dl['bytes'].sum()/1e6:.2f} MB")
    print(f"     UL: {len(ul):,} bursts  {ul['synthetic_flow_id'].nunique()} flows  "
          f"{ul['bytes'].sum()/1e6:.2f} MB")
    print()

print(f"  ✅  Cell 1 complete — {len(APPS)} app(s) loaded")


  scenario_config.yaml not found — discovered apps: ['aparat', 'filimo', 'igap', 'telegram', 'youtube']

  LOADING BURST POOLS
  📱 APARAT
     DL: 946 bursts  59 flows  28.75 MB
     UL: 94 bursts  46 flows  0.09 MB

  📱 FILIMO
     DL: 727 bursts  52 flows  29.22 MB
     UL: 83 bursts  28 flows  0.25 MB

  📱 IGAP
     DL: 693 bursts  107 flows  26.53 MB
     UL: 30 bursts  17 flows  0.06 MB

  📱 TELEGRAM
     DL: 703 bursts  21 flows  15.37 MB
     UL: 330 bursts  14 flows  1.85 MB

  📱 YOUTUBE
     DL: 2,185 bursts  67 flows  57.83 MB
     UL: 1,436 bursts  39 flows  3.17 MB

  ✅  Cell 1 complete — 5 app(s) loaded


## Cell 2 — Data Quality Audit

Every flag is classified as either:

- **MGEN edge-case** — statistically valid in the Markov model but requires
  conversion before MGEN can execute it. The upstream generator explicitly
  preserves these states (e.g. `is_zero_on`, first-burst `off_dur_s = NaN`).

- **Data corruption** — should not appear; indicates an upstream bug.
  These rows are **dropped**.


In [2]:
def get_corruption_mask(df: pd.DataFrame) -> pd.Series:
    """Rows that are unrecoverably broken and must be dropped."""
    return (
        df["on_dur_s"].isna()
        | (df["on_dur_s"] < 0)
        | (df["bytes"] <= 0)
        | (df["packets"] <= 0)
        | (df["off_dur_s"].fillna(0) < 0)
    )


def audit(df: pd.DataFrame, label: str) -> None:
    n        = len(df)
    pkt_size = (df["bytes"] / df["packets"].replace(0, np.nan)).fillna(0)

    # (display_name, mask, classification, action)
    checks = [
        ("on_dur_s == 0",
         df["on_dur_s"] == 0,
         "MGEN edge-case",
         "clamp to MIN_ON_DUR_S"),

        ("on_dur_s is NaN",
         df["on_dur_s"].isna(),
         "Data corruption",
         "drop row"),

        ("on_dur_s < 0",
         df["on_dur_s"] < 0,
         "Data corruption",
         "drop row"),

        ("off_dur_s is NaN  [first burst]",
         df["off_dur_s"].isna(),
         "MGEN edge-case",
         "fill 0.0  ← in assumption note"),

        ("off_dur_s < 0",
         df["off_dur_s"].fillna(0) < 0,
         "Data corruption",
         "drop row"),

        ("bytes <= 0",
         df["bytes"] <= 0,
         "Data corruption",
         "drop row"),

        ("packets <= 0",
         df["packets"] <= 0,
         "Data corruption",
         "drop row"),

        ("pkt_size > 1472  [raw]",
         pkt_size > 1472,
         "MGEN edge-case",
         "increase packet count (in Cell 7)"),

        ("pkt_size < 64   [raw]",
         pkt_size < 64,
         "MGEN edge-case",
         "pkt_size_export clipped to 64"),
    ]

    corruption = pd.Series(False, index=df.index)

    print(f"\n{'='*76}")
    print(f"  {label}  ({n:,} bursts, {df['synthetic_flow_id'].nunique()} flows)")
    print(f"{'='*76}")
    print(f"\n  {'Flag':<36} {'Count':>7}  {'%':>6}  "
          f"{'Class':<16}  Action")
    print(f"  {'-'*36} {'-'*7}  {'-'*6}  {'-'*16}  {'-'*30}")

    for name, mask, cls, action in checks:
        count = int(mask.sum())
        pct   = 100 * count / n if n else 0
        icon  = "⚠️" if count > 0 else "✓ "
        print(f"  {icon} {name:<35} {count:>7,}  {pct:>5.1f}%  "
              f"{cls:<16}  {action}")
        if cls == "Data corruption":
            corruption |= mask

    usable = df[~corruption]
    print(f"\n  Rows dropped (corruption) : {corruption.sum():,}")
    print(f"  Usable rows               : {len(usable):,} / {n:,}  "
          f"({100*len(usable)/n:.1f}%)")
    print(f"  Usable flows              : "
          f"{usable['synthetic_flow_id'].nunique()} / "
          f"{df['synthetic_flow_id'].nunique()}")


print("="*76)
print("  DATA QUALITY AUDIT — ALL APPS")
print("="*76)

for app in APPS:
    dl = app_raw[app]["dl"]
    ul = app_raw[app]["ul"]
    audit(dl, f"{app.upper()} — DOWNLINK")
    audit(ul, f"{app.upper()} — UPLINK")

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ASSUMPTION NOTE — off_dur_s NaN → 0.0
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  NaN in off_dur_s means this is the FIRST burst of its synthetic flow (no preceding idle gap exists in the model).
  This is statistically correct — the upstream generator explicitly preserves first-burst NaN.
  Filling with 0.0 is the correct MGEN default, but it carries a hidden assumption:

  ► Every synthetic flow starts immediately at its assigned start time.
  ► Notebook 1 MUST add an explicit per-flow random start offset when building the scenario timeline.
    Without this, all flows that fill the entire simulation window start at t=0 simultaneously, producing a traffic spike 
    at t=0 followed by silence — which looks like broken traffic during testing.
  ► See the required code change printed in Cell 8.

  This is only an execution-preparation projection, not a data correction.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

  DATA QUALITY AUDIT — ALL APPS

  APARAT — DOWNLINK  (946 bursts, 59 flows)

  Flag                                   Count       %  Class             Action
  ------------------------------------ -------  ------  ----------------  ------------------------------
  ⚠️ on_dur_s == 0                           224   23.7%  MGEN edge-case    clamp to MIN_ON_DUR_S
  ✓  on_dur_s is NaN                           0    0.0%  Data corruption   drop row
  ✓  on_dur_s < 0                              0    0.0%  Data corruption   drop row
  ⚠️ off_dur_s is NaN  [first burst]          59    6.2%  MGEN edge-case    fill 0.0  ← in assumption note
  ✓  off_dur_s < 0                             0    0.0%  Data corruption   drop row
  ✓  bytes <= 0                                0    0.0%  Data corruption   drop row
  ✓  packets <= 0                              0    0.0%  Data corruption   drop row
  ⚠️ pkt_size > 1472  [raw]                   18    1.9%  MGEN edge-case    increase packet count (in Cell

## Cell 3 — Zero-Duration Burst Analysis

Zero `on_dur_s` bursts are **not data corruption**. They are single-packet
or near-instant bursts that the segmentation logic preserved intentionally.
The upstream generator tracks `is_zero_on` and keeps these rather than
discarding them.

For MGEN they are an **execution edge-case**: MGEN cannot send over zero
duration, so we must clamp. This cell checks whether the chosen clamp
value keeps packet rates realistic.

In [3]:
# Raise to 0.010 if the pps warning fires below

MIN_ON_DUR_S = 0.001   # 1 ms minimum burst duration

def zero_dur_analysis(df: pd.DataFrame, label: str) -> None:
    z  = df[df["on_dur_s"] == 0].copy()
    nz = df[df["on_dur_s"] > 0].copy()

    print(f"\n{'='*76}")
    print(f"  {label} — zero-duration burst analysis")
    print(f"{'='*76}")
    print(f"  Classification : MGEN execution edge-case (not data corruption)")
    print(f"  Zero-duration  : {len(z):,}  ({100*len(z)/len(df):.1f}%)")
    print(f"  Non-zero       : {len(nz):,}  ({100*len(nz)/len(df):.1f}%)")

    if len(z) > 0:
        print(f"\n  Content of zero-duration bursts:")
        print(f"    bytes   : min={z.bytes.min():.0f}  "
              f"median={z.bytes.median():.0f}  max={z.bytes.max():.0f}")
        print(f"    packets : min={z.packets.min():.0f}  "
              f"median={z.packets.median():.0f}  max={z.packets.max():.0f}")
        print(f"    clusters: {sorted(z.cluster.unique().tolist())}")

        simulated_pps = z["packets"] / MIN_ON_DUR_S
        print(f"\n  If clamped to {MIN_ON_DUR_S*1000:.0f} ms  "
              f"(MIN_ON_DUR_S = {MIN_ON_DUR_S}):")
        print(f"    median pps : {simulated_pps.median():,.0f}")
        print(f"    max pps    : {simulated_pps.max():,.0f}")

        PPS_WARN = 100_000
        if simulated_pps.max() > PPS_WARN:
            safer = MIN_ON_DUR_S * 10
            print(f"\n    ⚠️  Max pps {simulated_pps.max():,.0f} exceeds "
                  f"{PPS_WARN:,} — unrealistically high.")
            print(f"       Set MIN_ON_DUR_S = {safer} "
                  f"({safer*1000:.0f} ms) at the top of this cell.")
            print(f"       That gives max pps = "
                  f"{z['packets'].max() / safer:,.0f}")
        else:
            print(f"    ✅  pps is within a realistic range — "
                  f"{MIN_ON_DUR_S*1000:.0f} ms clamp is safe.")

    if len(nz) > 0:
        print(f"\n  Non-zero on_dur_s distribution (seconds):")
        print(nz["on_dur_s"].describe()
              .apply(lambda x: f"{x:.6f}").to_string())


print("="*76)
print("  ZERO-DURATION BURST ANALYSIS — ALL APPS")
print("="*76)

for app in APPS:
    dl = app_raw[app]["dl"]
    ul = app_raw[app]["ul"]
    zero_dur_analysis(dl, f"{app.upper()} — DOWNLINK")
    zero_dur_analysis(ul, f"{app.upper()} — UPLINK")

  ZERO-DURATION BURST ANALYSIS — ALL APPS

  APARAT — DOWNLINK — zero-duration burst analysis
  Classification : MGEN execution edge-case (not data corruption)
  Zero-duration  : 224  (23.7%)
  Non-zero       : 722  (76.3%)

  Content of zero-duration bursts:
    bytes   : min=91  median=1411  max=1447
    packets : min=1  median=1  max=1
    clusters: [0, 1]

  If clamped to 1 ms  (MIN_ON_DUR_S = 0.001):
    median pps : 1,000
    max pps    : 1,000
    ✅  pps is within a realistic range — 1 ms clamp is safe.

  Non-zero on_dur_s distribution (seconds):
count    722.000000
mean       0.037567
std        0.041106
min        0.000032
25%        0.007801
50%        0.024368
75%        0.051939
max        0.303394

  APARAT — UPLINK — zero-duration burst analysis
  Classification : MGEN execution edge-case (not data corruption)
  Zero-duration  : 67  (71.3%)
  Non-zero       : 27  (28.7%)

  Content of zero-duration bursts:
    bytes   : min=83  median=87  max=1141
    packets : min=1  me

In [4]:
MAX_PKT = 1472   # UDP payload limit (bytes)
MIN_PKT = 64     # minimum sensible UDP payload

def pkt_size_analysis(df: pd.DataFrame, label: str) -> None:
    # only look at rows with valid volume
    clean    = df[(df["bytes"] > 0) & (df["packets"] > 0)].copy()
    raw_size = clean["bytes"] / clean["packets"]

    above = (raw_size > MAX_PKT).sum()
    below = (raw_size < MIN_PKT).sum()
    ok    = len(clean) - above - below

    print(f"\n{'='*76}")
    print(f"  {label} — raw implied packet size (bytes / packets)")
    print(f"{'='*76}")
    print(f"  min    : {raw_size.min():.1f} B")
    print(f"  median : {raw_size.median():.1f} B")
    print(f"  mean   : {raw_size.mean():.1f} B")
    print(f"  max    : {raw_size.max():.1f} B")
    print(f"\n  In range [{MIN_PKT}, {MAX_PKT}] : "
          f"{ok:,}  ({100*ok/len(clean):.1f}%)")
    print(f"  > {MAX_PKT} (needs packet-count fix) : "
          f"{above:,}  ({100*above/len(clean):.1f}%)")
    print(f"  < {MIN_PKT} (will be clipped to 64) : "
          f"{below:,}  ({100*below/len(clean):.1f}%)")

    if above > 0:
        oversized = raw_size[raw_size > MAX_PKT]
        # show what the old clip approach would have lost
        original_bytes  = clean.loc[oversized.index, "bytes"].sum()
        clipped_bytes   = (
            clean.loc[oversized.index, "packets"] * MAX_PKT
        ).sum()
        byte_loss_pct   = 100 * (1 - clipped_bytes / original_bytes)

        # show what the new approach preserves
        pkts_needed = np.ceil(
            clean.loc[oversized.index, "bytes"] / MAX_PKT
        ).astype(int)
        new_pkt_size = np.ceil(
            clean.loc[oversized.index, "bytes"] / pkts_needed
        ).astype(int)
        new_bytes = (pkts_needed * new_pkt_size).sum()
        new_error_pct = 100 * abs(new_bytes - original_bytes) / original_bytes

        print(f"\n  Impact comparison for the {above:,} oversized bursts:")
        print(f"  ┌─────────────────────────────────────────────────────┐")
        print(f"  │  Old approach (clip pkt_size to {MAX_PKT})             │")
        print(f"  │    Original bytes  : {original_bytes:>12,.0f}                │")
        print(f"  │    After clip      : {clipped_bytes:>12,.0f}                │")
        print(f"  │    Byte loss       : {byte_loss_pct:>11.1f}%                │")
        print(f"  ├─────────────────────────────────────────────────────┤")
        print(f"  │  New approach (increase packet count)               │")
        print(f"  │    Original bytes  : {original_bytes:>12,.0f}                │")
        print(f"  │    After adjust    : {new_bytes:>12,.0f}                │")
        print(f"  │    Byte error      : {new_error_pct:>11.2f}%  (≤ 1 B/pkt)  │")
        print(f"  └─────────────────────────────────────────────────────┘")


print("="*76)
print("  PACKET SIZE ANALYSIS — ALL APPS")
print("="*76)

for app in APPS:
    dl = app_raw[app]["dl"]
    ul = app_raw[app]["ul"]
    pkt_size_analysis(dl, f"{app.upper()} — DOWNLINK")
    pkt_size_analysis(ul, f"{app.upper()} — UPLINK")

  PACKET SIZE ANALYSIS — ALL APPS

  APARAT — DOWNLINK — raw implied packet size (bytes / packets)
  min    : 90.9 B
  median : 1413.8 B
  mean   : 1363.1 B
  max    : 1589.0 B

  In range [64, 1472] : 928  (98.1%)
  > 1472 (needs packet-count fix) : 18  (1.9%)
  < 64 (will be clipped to 64) : 0  (0.0%)

  Impact comparison for the 18 oversized bursts:
  ┌─────────────────────────────────────────────────────┐
  │  Old approach (clip pkt_size to 1472)             │
  │    Original bytes  :      168,008                │
  │    After clip      :      164,864                │
  │    Byte loss       :         1.9%                │
  ├─────────────────────────────────────────────────────┤
  │  New approach (increase packet count)               │
  │    Original bytes  :      168,008                │
  │    After adjust    :      168,072                │
  │    Byte error      :        0.04%  (≤ 1 B/pkt)  │
  └─────────────────────────────────────────────────────┘

  APARAT — UPLINK — raw imp

## Cell 5 — Inter-Burst Gap Analysis

Checks `off_dur_s` after NaN is resolved.
Long gaps (> 30 s) leave a UE silently idle mid-simulation — not a bug,
but it can look like broken traffic in a short test.

In [5]:
def gap_analysis(df: pd.DataFrame, label: str) -> None:
    gaps_real = df["off_dur_s"].dropna()

    print(f"\n{'='*76}")
    print(f"  {label} — inter-burst gap (off_dur_s)")
    print(f"{'='*76}")
    print(f"  NaN (first burst of flow)     : "
          f"{df['off_dur_s'].isna().sum():,}  → will become 0.0")
    print(f"  Non-NaN gaps (between bursts) : {len(gaps_real):,}")

    if len(gaps_real) > 0:
        print(f"\n  Between-burst gap distribution (seconds):")
        print(gaps_real.describe()
              .apply(lambda x: f"{x:.4f}").to_string())

        long_gaps = gaps_real[gaps_real > 30]
        if len(long_gaps) > 0:
            print(f"\n  ⚠️  {len(long_gaps)} gaps > 30 s — UE will be "
                  f"silent during these.")
            print(f"     Max gap: {long_gaps.max():.1f} s")
            print(f"     Not an error, but may look broken in short tests.")
        else:
            print(f"\n  ✅  No gaps > 30 s.")


print("="*76)
print("  INTER-BURST GAP ANALYSIS — ALL APPS")
print("="*76)

for app in APPS:
    dl = app_raw[app]["dl"]
    ul = app_raw[app]["ul"]
    gap_analysis(dl, f"{app.upper()} — DOWNLINK")
    gap_analysis(ul, f"{app.upper()} — UPLINK")

  INTER-BURST GAP ANALYSIS — ALL APPS

  APARAT — DOWNLINK — inter-burst gap (off_dur_s)
  NaN (first burst of flow)     : 59  → will become 0.0
  Non-NaN gaps (between bursts) : 887

  Between-burst gap distribution (seconds):
count    887.0000
mean       0.5594
std        3.6754
min        0.0172
25%        0.0293
50%        0.0396
75%        0.0682
max       60.3050

  ⚠️  3 gaps > 30 s — UE will be silent during these.
     Max gap: 60.3 s
     Not an error, but may look broken in short tests.

  APARAT — UPLINK — inter-burst gap (off_dur_s)
  NaN (first burst of flow)     : 46  → will become 0.0
  Non-NaN gaps (between bursts) : 48

  Between-burst gap distribution (seconds):
count     48.0000
mean      14.0707
std       51.2204
min        0.2882
25%        0.5323
50%        0.9438
75%        4.3115
max      339.1314

  ⚠️  4 gaps > 30 s — UE will be silent during these.
     Max gap: 339.1 s
     Not an error, but may look broken in short tests.

  FILIMO — DOWNLINK — inter-burst

## Cell 6 — Per-Flow Distribution & UE Capacity Estimate

Shows how bursts and bytes are distributed across flows, then estimates
how many UEs the pool can realistically support.


In [6]:
# ── thresholds (tune as needed) ───────────────────────────────────────────────
MIN_BURSTS_PER_FLOW = 2    # flows below this are "thin" and excluded
MIN_BURSTS_PER_UE   = 10   # each UE must receive at least this many DL bursts
TARGET_N_UE         = 6    # desired UE count

# ── per-flow distribution ─────────────────────────────────────────────────────

def flow_distribution(df: pd.DataFrame, label: str) -> pd.DataFrame:
    stats = (
        df.groupby("synthetic_flow_id")
          .agg(
              n_bursts    = ("synthetic_burst_idx", "count"),
              total_bytes = ("bytes",               "sum"),
              active_s    = ("on_dur_s",            "sum"),
              idle_s      = ("off_dur_s",           "sum"),
              cluster     = ("cluster",
                             lambda x: x.mode().iat[0]),
          )
          .assign(
              idle_s  = lambda d: d["idle_s"].fillna(0),
              total_s = lambda d: d["active_s"] + d["idle_s"],
              MB      = lambda d: d["total_bytes"] / 1e6,
          )
    )

    print(f"\n{'='*76}")
    print(f"  {label} — per-flow distribution")
    print(f"{'='*76}")
    print(f"  Flows                       : {len(stats)}")
    print(f"  Bursts  min/median/mean/max : "
          f"{stats.n_bursts.min()} / "
          f"{stats.n_bursts.median():.0f} / "
          f"{stats.n_bursts.mean():.1f} / "
          f"{stats.n_bursts.max()}")
    print(f"  MB      min/median/mean/max : "
          f"{stats.MB.min():.3f} / "
          f"{stats.MB.median():.3f} / "
          f"{stats.MB.mean():.3f} / "
          f"{stats.MB.max():.2f}")

    bins   = [0, 1, 3, 10, 50, 100_000]
    labels = ["1", "2-3", "4-10", "11-50", "51+"]
    stats["burst_bucket"] = pd.cut(
        stats["n_bursts"], bins=bins, labels=labels, right=True
    )
    print("\n  Flows by burst-count bucket:")
    for bucket, count in (
        stats["burst_bucket"].value_counts().sort_index().items()
    ):
        pct = 100 * count / len(stats)
        bar = "█" * int(pct / 2)
        print(f"    {str(bucket):<8}  {count:>4}  ({pct:5.1f}%)  {bar}")

    if "cluster" in df.columns:
        print("\n  Bursts and volume by cluster:")
        clu = df.groupby("cluster").agg(
            bursts = ("synthetic_burst_idx", "count"),
            MB     = ("bytes", lambda x: x.sum() / 1e6),
        )
        clu["%_bursts"] = (
            100 * clu["bursts"] / clu["bursts"].sum()
        ).round(1)
        clu["%_MB"] = (
            100 * clu["MB"] / clu["MB"].sum()
        ).round(1)
        print(clu.to_string(float_format="{:.2f}".format))

    return stats



# ── UE capacity estimate ──────────────────────────────────────────────────────

def usable_pool(df: pd.DataFrame, min_bursts: int) -> pd.DataFrame:
    clean    = df[~get_corruption_mask(df)]
    per_flow = (
        clean.groupby("synthetic_flow_id")
             .agg(n_bursts=("synthetic_burst_idx", "count"),
                  MB       =("bytes", lambda x: x.sum() / 1e6))
    )
    return per_flow[per_flow["n_bursts"] >= min_bursts]


def build_namespaced_pool(direction: str) -> pd.DataFrame:
    """
    Combine all apps for one direction with unique flow identity.
    synthetic_flow_id is only unique within one app — namespace it as
    <app>::<flow_id> before concatenating so capacity estimates are correct.
    """
    frames = []
    for app in APPS:
        df = app_raw[app][direction].copy()
        if df.empty:
            continue
        df["synthetic_flow_id"] = (
            app + "::" + df["synthetic_flow_id"].astype(str)
        )
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


dl_all = build_namespaced_pool("dl")
ul_all = build_namespaced_pool("ul")

dl_pool = usable_pool(dl_all, MIN_BURSTS_PER_FLOW)
ul_pool = usable_pool(ul_all, MIN_BURSTS_PER_FLOW)

total_dl = int(dl_pool["n_bursts"].sum())
total_ul = int(ul_pool["n_bursts"].sum())
max_ues  = total_dl // MIN_BURSTS_PER_UE if total_dl > 0 else 0

print(f"\n{'='*76}")
print(f"  UE CAPACITY ESTIMATE")
print(f"{'='*76}")
print(f"  min bursts per flow  : {MIN_BURSTS_PER_FLOW}")
print(f"  min DL bursts per UE : {MIN_BURSTS_PER_UE}")
print(f"  target N_UE          : {TARGET_N_UE}")
print(f"\n  DL usable flows  : {len(dl_pool)} / {dl_all['synthetic_flow_id'].nunique() if not dl_all.empty else 0}")
print(f"  DL usable bursts : {total_dl:,}")
print(f"  UL usable flows  : {len(ul_pool)} / {ul_all['synthetic_flow_id'].nunique() if not ul_all.empty else 0}")
print(f"  UL usable bursts : {total_ul:,}")
print(f"\n  Max supportable UEs (DL) : {max_ues}")

if max_ues >= TARGET_N_UE:
    recommended = TARGET_N_UE
    print(f"  ✅  Target of {TARGET_N_UE} UEs is achievable.")
else:
    recommended = max_ues
    print(f"  ⚠️  Only {max_ues} UEs can get ≥{MIN_BURSTS_PER_UE} DL bursts.")
    print(f"     Reduce TARGET_N_UE to {max_ues} OR lower MIN_BURSTS_PER_UE.")

# store for Cell 8
_recommended_ue = recommended

if recommended == 0:
    print("\n  ❌  recommended UEs = 0 — the burst pools may be empty or too sparse.")
    print("     Check that Notebook 0 ran successfully and artifacts/ is populated.")
    print("     Lower MIN_BURSTS_PER_FLOW or MIN_BURSTS_PER_UE and re-run.")
    _dl_fpu  = (1, 2)
    _ul_fpu  = (1, 1)
    heavy_n  = 0
    medium_n = 0
    light_n  = 0
    _class_dist = {"heavy": 0, "medium": 0, "light": 0}
else:
    _dl_fpu = (max(1, len(dl_pool)//recommended - 1),
               max(2, len(dl_pool)//recommended + 2))
    _ul_fpu = (max(1, len(ul_pool)//recommended),
               max(1, len(ul_pool)//recommended + 1))
    heavy_n  = 2
    light_n  = 1
    medium_n = max(1, recommended - heavy_n - light_n)
    _class_dist = {"heavy": heavy_n, "medium": medium_n, "light": light_n}

print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Recommended settings → Notebook 1, Cell 2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  N_UE                        = {recommended}
  FLOWS_PER_USER_DL (range)   = {_dl_fpu}
  FLOWS_PER_USER_UL (range)   = {_ul_fpu}
  USER_CLASS_DISTRIBUTION     = {{'heavy': {heavy_n}, 'medium': {medium_n}, 'light': {light_n}}}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print("="*76)
print("  PER-FLOW DISTRIBUTION & UE CAPACITY — ALL APPS")
print("="*76)

_recommended_ue_per_app = {}
for app in APPS:
    dl = app_raw[app]["dl"]
    ul = app_raw[app]["ul"]
    flow_distribution(dl, f"{app.upper()} — DOWNLINK")
    flow_distribution(ul, f"{app.upper()} — UPLINK")
    # store recommended UE count for Cell 8
    _dl_good = dl.groupby("synthetic_flow_id").size()
    _dl_good = _dl_good[_dl_good >= MIN_BURSTS_PER_FLOW]
    # Do NOT clamp to max(1,...) — zero is a valid result meaning
    # this app's pool cannot support even one UE at MIN_BURSTS_PER_UE.
    _ue_cap  = len(_dl_good) * MIN_BURSTS_PER_FLOW // MIN_BURSTS_PER_UE
    _recommended_ue_per_app[app] = min(_ue_cap, TARGET_N_UE)

print("\n  UE capacity summary:")
for app, n in _recommended_ue_per_app.items():
    print(f"    {app:<14} → recommended N_UE ≤ {n}")
_recommended_ue = min(_recommended_ue_per_app.values())
print(f"\n  Conservative N_UE (min across apps): {_recommended_ue}")



  DOWNLINK — per-flow distribution
  Flows                       : 67
  Bursts  min/median/mean/max : 1 / 6 / 32.6 / 292
  MB      min/median/mean/max : 0.000 / 0.070 / 0.863 / 7.87

  Flows by burst-count bucket:
    1           15  ( 22.4%)  ███████████
    2-3         11  ( 16.4%)  ████████
    4-10        13  ( 19.4%)  █████████
    11-50       15  ( 22.4%)  ███████████
    51+         13  ( 19.4%)  █████████

  Bursts and volume by cluster:
         bursts    MB  %_bursts  %_MB
cluster                              
0          1208 57.36     55.30 99.20
1            34  0.12      1.60  0.20
2            32  0.03      1.50  0.10
3           704  0.06     32.20  0.10
4           207  0.26      9.50  0.50

  UPLINK — per-flow distribution
  Flows                       : 39
  Bursts  min/median/mean/max : 1 / 25 / 36.8 / 237
  MB      min/median/mean/max : 0.000 / 0.060 / 0.081 / 0.35

  Flows by burst-count bucket:
    1            4  ( 10.3%)  █████
    2-3          8  ( 20.5%)  ███

## Cell 7 — MGEN-Feasibility Projection (Cleaned Export)

Projects synthetic bursts into MGEN-executable bursts.


| Step | Input state | Output | Classification |
|---|---|---|---|
| Drop | bad bytes / pkts / negative dur | row removed | Data corruption |
| Fill gap | `off_dur_s = NaN` | `0.0` | MGEN edge-case projection |
| Clamp duration | `on_dur_s = 0` | `MIN_ON_DUR_S` | MGEN edge-case projection |
| Packet count fix | `bytes/packets > 1472` | increase `packets_export` | Improvement |
| Derive pkt_size | `ceil(bytes / packets_export)` | clipped to [64, 1472] | Approximation |
| Derive pps | `packets_export / on_dur_s` | after clamped duration | Derived |

### Improvement 1 — packet count adjustment

The approach increases packet count to the minimum needed to carry
the original bytes within the UDP limit:

```python
packets_export = max(packets_original, ceil(bytes / 1472))
pkt_size_export = ceil(bytes / packets_export)
```

 example: `ceil(10000/1472) = 7` → `packets_export = max(3,7) = 7`
→ `pkt_size_export = ceil(10000/7) = 1429` → MGEN sends `7×1429 = 10003`
bytes. Residual error ≤ 1 byte per packet, which is negligible.

The trade-off: packet count (and thus pps) may increase for oversized
bursts, but byte fidelity is preserved — which matters more for load
generation.

In [7]:
def project_to_mgen(
    df: pd.DataFrame,
    label: str,
    min_on_dur_s: float,
    max_pkt: int = 1472,
    min_pkt: int = 64,
) -> pd.DataFrame:
    """
    Project synthetic bursts into MGEN-executable bursts.

    Parameters
    ----------
    df           : raw synthetic burst DataFrame
    label        : display label for logging
    min_on_dur_s : minimum burst duration after clamping (seconds)
    max_pkt      : UDP payload ceiling (bytes)
    min_pkt      : UDP payload floor  (bytes)
    """
    out = df.copy()
    n   = len(out)

    
    # ── 1. drop data corruption ───────────────────────────────────────────────
    bad     = get_corruption_mask(out)
    out     = out[~bad].copy()
    dropped = int(bad.sum())

    
    # ── 2. project first-burst NaN gap → 0.0 ─────────────────────────────────
    #    Assumption: flow starts immediately at its assigned start time.
    #    Notebook 1 MUST add a per-flow random start offset.
    
    nan_off = int(out["off_dur_s"].isna().sum())
    out["off_dur_s"] = out["off_dur_s"].fillna(0.0)

    
    # ── 3. clamp zero-duration bursts ────────────────────────────────────────
    #    Edge-case, not corruption: single-packet near-instant bursts.
    
    zero_dur = int((out["on_dur_s"] == 0).sum())
    out["on_dur_s"] = out["on_dur_s"].clip(lower=min_on_dur_s)

    
    # ── 4. packet count adjustment ────────────────────────────
    #    Instead of clipping pkt_size (losing bytes), we increase packet
    #    count to the minimum needed to stay within max_pkt.
    #
    #    packets_export = max(packets_original, ceil(bytes / max_pkt))
    #    pkt_size_export = ceil(bytes / packets_export)
    #
    #    Residual byte error <= 1 byte per packet (from ceil rounding).

    
    pkts_needed = np.ceil(out["bytes"] / max_pkt).astype(int)
    out["packets_export"] = np.maximum(
        out["packets"].astype(int), pkts_needed
    )

    pkt_count_increased = int(
        (out["packets_export"] > out["packets"]).sum()
    )

    
    # ── 5. derive pkt_size_export ─────────────────────────────────────────────
    #    ceil to minimise byte undercounting 
    
    out["pkt_size_raw"]    = out["bytes"] / out["packets"]   # original, for reference
    out["pkt_size_export"] = np.ceil(
        out["bytes"] / out["packets_export"]
    ).astype(int)

    
    # clip the floor only (ceil already guarantees <= max_pkt + 1 rounding)
    
    clipped_lo = int((out["pkt_size_export"] < min_pkt).sum())
    out["pkt_size_export"] = out["pkt_size_export"].clip(
        lower=min_pkt, upper=max_pkt
    )

    
    # ── 6. derive pps from projected packets and clamped duration ─────────────
    
    out["pps"] = (out["packets_export"] / out["on_dur_s"]).round(3)

    # ── byte fidelity report ─────────────────────────────────────────────────
    
    original_bytes  = out["bytes"].sum()
    projected_bytes = (out["packets_export"] * out["pkt_size_export"]).sum()
    byte_error_pct  = 100 * abs(projected_bytes - original_bytes) / original_bytes

    print(f"\n  {label}")
    print(f"    Original rows                  : {n:,}")
    print(f"    Dropped (data corruption)      : {dropped:,}")
    print(f"    NaN off_dur_s → 0.0            : {nan_off:,}   "
          f"(first-burst projection)")
    print(f"    on_dur_s=0 → {min_on_dur_s*1000:.0f} ms            : "
          f"{zero_dur:,}   (MGEN edge-case projection)")
    print(f"    Packet count increased         : {pkt_count_increased:,}   "
          f"(byte-preserving fix)")
    print(f"    pkt_size_export clipped lo (<{min_pkt}): {clipped_lo:,}")
    print(f"    Final rows                     : {len(out):,}")
    print(f"    Final flows                    : "
          f"{out['synthetic_flow_id'].nunique()}")
    print(f"    Byte fidelity error            : {byte_error_pct:.3f}%  "
          f"(≤ 1 B per packet from ceil)")

    return out


print("="*76)
print("  PROJECTING TO MGEN-FEASIBLE BURSTS")
print("="*76)

print("="*76)
print("  MGEN-FEASIBILITY PROJECTION — ALL APPS")
print("="*76)

# app_clean[app] = {"dl": cleaned_df, "ul": cleaned_df}
app_clean = {}

for app in APPS:
    dl = app_raw[app]["dl"]
    ul = app_raw[app]["ul"]
    print(f"\n  {'─'*60}")
    print(f"  📱 {app.upper()}")
    print(f"  {'─'*60}")
    dl_clean = project_to_mgen(dl, f"{app.upper()} DOWNLINK", MIN_ON_DUR_S)
    ul_clean = project_to_mgen(ul, f"{app.upper()} UPLINK",   MIN_ON_DUR_S)
    app_clean[app] = {"dl": dl_clean, "ul": ul_clean}

print("\n  ✅  Cell 7 complete — all apps projected")

# ── save — distinct projected artifact per app ────────────────────────────────
# Writes synth_test_bursts_mgen.parquet alongside the original
# synth_test_bursts_markov.parquet — raw Markov output is NEVER overwritten.
#
# Notebook 1 prefers synth_test_bursts_mgen.parquet when present,
# falling back to synth_test_bursts_markov.parquet automatically.
new_cols = ["packets_export", "pkt_size_raw", "pkt_size_export", "pps"]

for app in APPS:
    dl_run = find_latest_run_dir(ART / app / "downlink")
    ul_run = find_latest_run_dir(ART / app / "uplink")

    dl_out = dl_run / "data" / "synth_test_bursts_mgen.parquet"
    ul_out = ul_run / "data" / "synth_test_bursts_mgen.parquet"

    app_clean[app]["dl"].to_parquet(dl_out, index=False)
    app_clean[app]["ul"].to_parquet(ul_out, index=False)

    print(f"  ✅  {app.upper()}")
    print(f"       DL → {dl_out.relative_to(ROOT)}")
    print(f"       UL → {ul_out.relative_to(ROOT)}")
    print(f"       (synth_test_bursts_markov.parquet preserved unchanged)")

print()
print("  Columns added to projected file:")
for c in new_cols:
    print(f"    {c}")

  PROJECTING TO MGEN-FEASIBLE BURSTS
  MGEN-FEASIBILITY PROJECTION — ALL APPS

  ────────────────────────────────────────────────────────────
  📱 APARAT
  ────────────────────────────────────────────────────────────

  APARAT DOWNLINK
    Original rows                  : 946
    Dropped (data corruption)      : 0
    NaN off_dur_s → 0.0            : 59   (first-burst projection)
    on_dur_s=0 → 1 ms            : 224   (MGEN edge-case projection)
    Packet count increased         : 18   (byte-preserving fix)
    pkt_size_export clipped lo (<64): 0
    Final rows                     : 946
    Final flows                    : 59
    Byte fidelity error            : 0.035%  (≤ 1 B per packet from ceil)

  APARAT UPLINK
    Original rows                  : 94
    Dropped (data corruption)      : 0
    NaN off_dur_s → 0.0            : 46   (first-burst projection)
    on_dur_s=0 → 1 ms            : 67   (MGEN edge-case projection)
    Packet count increased         : 0   (byte-preserving f

## Cell 8 — Final Summary & Notebook 1 Checklist

The config block to copy into Cell 2, plus the two required
code changes in Notebook 1 that complete the projection started here.

In [8]:
print("="*76)
print("  FINAL SUMMARY — ALL APPS")
print("="*76)

for app in APPS:
    dl_clean = app_clean[app]["dl"]
    ul_clean = app_clean[app]["ul"]
    dl_flows_n = dl_clean["synthetic_flow_id"].nunique()
    ul_flows_n = ul_clean["synthetic_flow_id"].nunique()
    dl_mb      = dl_clean["bytes"].sum() / 1e6
    ul_mb      = ul_clean["bytes"].sum() / 1e6
    fl_ratio   = dl_flows_n / ul_flows_n if ul_flows_n else 0

    print(f"\n  📱 {app.upper()}")
    print(f"     DL  flows: {dl_flows_n}   bursts: {len(dl_clean):,}   "
          f"volume: {dl_mb:.2f} MB")
    print(f"     UL  flows: {ul_flows_n}   bursts: {len(ul_clean):,}   "
          f"volume: {ul_mb:.2f} MB")
    print(f"     DL/UL byte ratio: {dl_mb/ul_mb:.1f}:1   "
          f"flow ratio: {fl_ratio:.2f}:1")

print()
print("━"*76)
print("  RECOMMENDED scenario_config.yaml  (paste and adjust as needed)")
print("━"*76)

# Build apps list lines
_apps_lines = "\n".join(f"  - {app}" for app in APPS)

# Build user class distribution — use conservative N_UE
import math as _math
_n = _recommended_ue
if _n == 0:
    _class_dist = {"heavy": 0, "medium": 0, "light": 0}
else:
    _heavy  = max(1, _math.floor(_n * 0.33))
    _light  = max(1, _math.floor(_n * 0.17))
    _medium = _n - _heavy - _light
    _class_dist = {"heavy": _heavy, "medium": _medium, "light": _light}
_class_lines = "\n".join(f"    {cls}: {n}" for cls, n in _class_dist.items())

# Per-app notes
_notes = []
for app in APPS:
    dl_clean = app_clean[app]["dl"]
    ul_clean = app_clean[app]["ul"]
    dl_mb    = dl_clean["bytes"].sum() / 1e6
    ul_mb    = ul_clean["bytes"].sum() / 1e6
    ratio    = dl_mb / ul_mb if ul_mb > 0 else float("inf")
    _notes.append(f"#   {app:<14}: DL/UL={ratio:.1f}:1  "
                  f"DL={dl_mb:.1f}MB  UL={ul_mb:.2f}MB")
_notes_block = "\n".join(_notes)

_yaml = f"""
# ── paste into scenario_config.yaml ──────────────────────────────────────────

simulation:
  n_ue: {_recommended_ue}
  duration: 600

apps:
{_apps_lines}

user_classes:
  distribution:
{_class_lines}

temporal_correlation:
  enabled: true
  rtt_delay_range: [0.010, 0.050]
  dl_bursts_per_ul_request: [1, 5]
  jitter: 0.005
  mode_overrides: {{}}

# ── Audit notes ───────────────────────────────────────────────────────────────
{_notes_block}
#   MIN_ON_DUR_S used: {MIN_ON_DUR_S}s ({MIN_ON_DUR_S*1000:.0f}ms)
#   Recommended N_UE : {_recommended_ue} (conservative — limited by smallest pool)
"""
print(_yaml)

print("━"*76)
print("  ✅  Notebook 0 complete — cleaned parquets are in app_clean[app]")
print("  Run Notebook 1 next (build_scenario_1.ipynb)")
print("━"*76)


  FINAL SUMMARY — ALL APPS

  📱 APARAT
     DL  flows: 59   bursts: 946   volume: 28.75 MB
     UL  flows: 46   bursts: 94   volume: 0.09 MB
     DL/UL byte ratio: 321.3:1   flow ratio: 1.28:1

  📱 FILIMO
     DL  flows: 52   bursts: 727   volume: 29.22 MB
     UL  flows: 28   bursts: 83   volume: 0.25 MB
     DL/UL byte ratio: 118.2:1   flow ratio: 1.86:1

  📱 IGAP
     DL  flows: 107   bursts: 693   volume: 26.53 MB
     UL  flows: 17   bursts: 30   volume: 0.06 MB
     DL/UL byte ratio: 465.2:1   flow ratio: 6.29:1

  📱 TELEGRAM
     DL  flows: 21   bursts: 703   volume: 15.37 MB
     UL  flows: 14   bursts: 330   volume: 1.85 MB
     DL/UL byte ratio: 8.3:1   flow ratio: 1.50:1

  📱 YOUTUBE
     DL  flows: 67   bursts: 2,185   volume: 57.83 MB
     UL  flows: 39   bursts: 1,436   volume: 3.17 MB
     DL/UL byte ratio: 18.2:1   flow ratio: 1.72:1

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  RECOMMENDED scenario_config.yaml  (paste and adjust as nee